# Evaluate Unified SFT Model (Hugging Face Cloud integration)

This notebook clones the latest code repository from GitHub and runs the evaluation pipeline. The dataset and fine-tuned model adapter are downloaded directly from the Hugging Face Hub, with no local requirements.

In [1]:
import os
import sys
import torch
from pathlib import Path

# Safe repo cloning and directory changing
repo_url = "https://github.com/NguyenAn05/exact-2026.git"
repo_dir = "exact-2026"

if Path(os.getcwd()).name == repo_dir:
    print(f"Already inside the repository directory: {os.getcwd()}")
else:
    if not os.path.exists(repo_dir):
        print(f"Cloning repository from {repo_url}...")
        os.system(f"git clone {repo_url}")
    os.chdir(repo_dir)
    print(f"Changed working directory to: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Changed working directory to: /workspace/exact-2026
Current working directory: /workspace/exact-2026
CUDA Available: True
GPU: AMD Instinct MI300X VF


In [2]:
import re
import json
import gc
import argparse
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login
import contextlib
import io

# Try to import rapidfuzz safely for soft similarity metrics
try:
    from rapidfuzz import fuzz
except ImportError:
    fuzz = None

# Try importing Z3
try:
    from z3.z3 import *
except ImportError:
    from z3 import *

In [3]:
def check_property(solver, prop):
    """
    Check a boolean property under the solver's constraints.
    Returns:
        "Yes" if the property is proven to be True (Not(prop) is unsat)
        "No" if the property is proven to be False (prop is unsat)
        "Unknown" otherwise.
    """
    # Set a timeout (2000ms) to prevent hanging
    solver.set("timeout", 2000)
    
    # Try to prove prop is True (by showing Not(prop) is inconsistent with constraints)
    solver.push()
    solver.add(Not(prop))
    res = solver.check()
    solver.pop()
    if res == unsat:
        return "Yes"
        
    # Try to prove prop is False (by showing prop is inconsistent with constraints)
    solver.push()
    solver.add(prop)
    res = solver.check()
    solver.pop()
    if res == unsat:
        return "No"
        
    return "Uncertain"


def check_mcq(solver, options_dict):
    """
    Evaluate multiple choice options under the solver's constraints.
    Returns:
        The key of the option (e.g., 'A', 'B', 'C', 'D') that is proven to be True.
        "Unknown" if none or multiple are proven.
    """
    # Set a timeout (2000ms) to prevent hanging
    solver.set("timeout", 2000)
    
    proved = []
    for opt, opt_expr in options_dict.items():
        solver.push()
        solver.add(Not(opt_expr))
        res = solver.check()
        solver.pop()
        if res == unsat:
            proved.append(opt)
            
    if len(proved) == 1:
        return proved[0]
    elif len(proved) > 1:
        return proved[0]
    return "Uncertain"


def run_z3_code(z3_code_str: str) -> str:
    """
    Execute a Z3 code block string and return the extracted Answer.
    """
    # Prepare the execution namespace
    globals_dict = {
        'check_mcq': check_mcq,
        'check_property': check_property,
        '__builtins__': __builtins__
    }
    
    # Try importing from z3.z3 (specific to this environment) or fallback to standard z3 module
    try:
        import z3.z3 as z3_mod
    except ImportError:
        try:
            import z3 as z3_mod
        except ImportError as e:
            return f"Error: Z3 module not found ({e})"
            
    # Inject Z3 symbols into globals_dict for execution
    for name in dir(z3_mod):
        if not name.startswith('__'):
            globals_dict[name] = getattr(z3_mod, name)
            
    # Redirect stdout to capture the printed answer
    stdout_capture = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(z3_code_str, globals_dict)
    except Exception as e:
        return f"Error: {e}"
        
    output = stdout_capture.getvalue()
    
    # Extract the answer from printed output (e.g., "Answer: A" or "Answer: Yes")
    match = re.search(r"Answer:\s*(\w+)", output, re.IGNORECASE)
    if match:
        return match.group(1)
        
    # Check for printed output style like "Answer, Yes" or "Answer: Yes" with multiple spaces
    match_comma = re.search(r"Answer\s*,\s*(\w+)", output, re.IGNORECASE)
    if match_comma:
        return match_comma.group(1)
        
    return "Uncertain"

In [4]:
def parse_xml_response(text: str) -> dict:
    """
    Parses the XML format generated by the Model (Singular version).
    """
    premises_fol = []
    fol_match = re.search(r"<premises_fol>(.*?)</premises_fol>", text, re.DOTALL)
    if fol_match:
        fol_content = fol_match.group(1).strip()
        for line in fol_content.split("\n"):
            line = line.strip()
            if not line:
                continue
            line = re.sub(r"^\d+\.\s*", "", line)
            premises_fol.append(line)
            
    questions = []
    q_match = re.search(r"<question_output>(.*?)</question_output>", text, re.DOTALL)
    if q_match:
        q_content = q_match.group(1).strip()
        
        used_match = re.search(r"<premises_used>(.*?)</premises_used>", q_content, re.DOTALL)
        premises_used = []
        if used_match:
            try:
                premises_used = json.loads(used_match.group(1).strip())
            except Exception:
                premises_used = [int(n) for n in re.findall(r"\d+", used_match.group(1))]
                
        z3_match = re.search(r"<z3_code>(.*?)</z3_code>", q_content, re.DOTALL)
        z3_code = z3_match.group(1).strip() if z3_match else ""
        
        ans_match = re.search(r"<answer>(.*?)</answer>", q_content, re.DOTALL)
        answer = ans_match.group(1).strip() if ans_match else ""
        
        exp_match = re.search(r"<explanation>(.*?)</explanation>", q_content, re.DOTALL)
        explanation = exp_match.group(1).strip() if exp_match else ""
        
        questions.append({
            "id": 1,
            "premises_used": premises_used,
            "z3_code": z3_code,
            "answer": answer,
            "explanation": explanation
        })
        
    return {
        "premises_fol": premises_fol,
        "questions": questions
    }

In [5]:
def parse_args():
    parser = argparse.ArgumentParser(description="Evaluate the logic reasoning model on the unified HF dataset.")
    parser.add_argument("--model_id", type=str, default="Qwen/Qwen2.5-7B-Instruct")
    parser.add_argument("--adapter_dir", type=str, default="NguyenAn05/qwen2.5-type1-sft-lora")
    parser.add_argument("--dataset_id", type=str, default="NguyenAn05/exact_2026_model1_fol_translator")
    parser.add_argument("--output_file", type=str, default="results/logic_eval_results.json")
    parser.add_argument("--sample_limit", type=int, default=-1)
    parser.add_argument("--max_new_tokens", type=int, default=2048)
    
    # Check if running in a Jupyter notebook/IPython session
    is_ipython = False
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            is_ipython = True
    except ImportError:
        pass
        
    if is_ipython or any('ipykernel' in arg for arg in sys.argv):
        return parser.parse_args(args=[])
    else:
        return parser.parse_args()

In [ ]:
args = parse_args()

# Login HF (required to download private datasets or adapters if applicable)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
login(token=HF_TOKEN)

# Load dataset from HF Hub
print(f"Loading dataset from Hugging Face Hub: {args.dataset_id}...")
dataset = load_dataset(args.dataset_id)
test_samples = list(dataset['test'])
    
if args.sample_limit > 0:
    test_samples = test_samples[:args.sample_limit]
    
print(f"Loaded {len(test_samples)} test samples successfully.")

# Load Tokenizer & Model
print(f"Loading tokenizer for {args.model_id}...")
tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
# Cấu hình padding bên TRÁI cực kỳ quan trọng cho batch inference của Causal LM
tokenizer.padding_side = 'left'

# Get Qwen im_end token ID
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

print(f"Loading base model {args.model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    args.model_id,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='flash_attention_2'
)

print(f"Loading LoRA adapter from Hugging Face Hub: {args.adapter_dir}...")
try:
    model = PeftModel.from_pretrained(base_model, args.adapter_dir)
except Exception as e:
    print(f"Warning: Could not load adapter from {args.adapter_dir} ({e}). Falling back to local directory or base model.")
    if os.path.exists(args.adapter_dir):
        model = PeftModel.from_pretrained(base_model, args.adapter_dir)
    else:
        model = base_model
    
model.eval()
device = next(model.parameters()).device

results = []
batch_size = 16
print(f"\n--- Running Model Inference (Batch Size = {batch_size}) ---")

# Vòng lặp nhảy theo kích thước batch_size
for i in tqdm(range(0, len(test_samples), batch_size), desc="Inference"):
    batch_samples = test_samples[i:i+batch_size]
    
    prompts = []
    gt_parsed_list = []
    user_prompts = []
    gt_assistants = []
    
    # Gom dữ liệu của cả batch
    for sample in batch_samples:
        messages = sample['messages']

        system_prompt = next((m['content'] for m in messages if m['role'] == 'system'), '')
        user_prompt = next((m['content'] for m in messages if m['role'] == 'user'), '')
        gt_assistant = next((m['content'] for m in messages if m['role'] == 'assistant'), '')

        # Parse ground truth
        gt_parsed = parse_xml_response(gt_assistant)
        gt_parsed_list.append(gt_parsed)
        user_prompts.append(user_prompt)
        gt_assistants.append(gt_assistant)

        prompt = tokenizer.apply_chat_template([
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ], tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)

    # Tokenize toàn bộ danh sách prompt trong batch (padding=True)
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(device)
    input_len = inputs.input_ids.shape[1] # Độ dài câu dài nhất sau khi đệm (pad)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=args.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
            use_cache=True,
            repetition_penalty=1.05
        )

    # Giải mã và lưu trữ kết quả cho từng mẫu trong batch
    for j, sample in enumerate(batch_samples):
        # Nhờ left-padding, phần text sinh mới của mọi câu đều bắt đầu từ index input_len
        gen_text = tokenizer.decode(outputs[j][input_len:], skip_special_tokens=True)
        llm_parsed = parse_xml_response(gen_text)

        results.append({
            'idx': i + j,
            'premises_text': user_prompts[j],
            'gt_assistant': gt_assistants[j],
            'gt_parsed': gt_parsed_list[j],
            'raw_generation': gen_text,
            'llm_parsed': llm_parsed
        })

# Free memory
print("\nFreeing GPU memory...")
del model
del base_model
gc.collect()
torch.cuda.empty_cache()

# Z3 Verification Layer
print("\n--- Running Z3 Symbolic Verification ---")
question_results = []

for res in tqdm(results, desc="Verification"):
    gt_parsed = res['gt_parsed']
    llm_parsed = res['llm_parsed']
    
    gt_fol = gt_parsed.get('premises_fol', [])
    llm_fol = llm_parsed.get('premises_fol', [])
    
    fol_exact_match = (llm_fol == gt_fol)
    
    if fuzz is not None:
        llm_fol_str = "\n".join(llm_fol)
        gt_fol_str = "\n".join(gt_fol)
        fol_similarity = fuzz.token_sort_ratio(llm_fol_str, gt_fol_str)
    else:
        fol_similarity = 100.0 if fol_exact_match else 0.0
        
    gt_questions = gt_parsed.get('questions', [])
    llm_questions = llm_parsed.get('questions', [])
    llm_q_map = {q['id']: q for q in llm_questions}
    
    for gt_q in gt_questions:
        q_id = gt_q['id']
        gt_ans = gt_q['answer']
        gt_used = gt_q['premises_used']
        gt_exp = gt_q.get('explanation', '')
        
        llm_q = llm_q_map.get(q_id, {})
        llm_ans = llm_q.get('answer', 'UNKNOWN')
        llm_used = llm_q.get('premises_used', [])
        llm_z3 = llm_q.get('z3_code', '')
        llm_exp = llm_q.get('explanation', '')
        
        z3_result = 'Error: Z3 code not found'
        error_type = 'SUCCESS'
        
        if llm_z3:
            try:
                z3_result = run_z3_code(llm_z3)
            except Exception as e:
                z3_result = f'Error: {e}'
        else:
            error_type = 'Z3_CODE_NOT_FOUND'
            
        is_z3_success = not str(z3_result).startswith('Error')
        
        if not llm_parsed.get('questions'):
            error_type = 'XML_PARSING_ERROR'
        elif error_type != 'Z3_CODE_NOT_FOUND' and not is_z3_success:
            err_msg = str(z3_result).lower()
            if 'syntax' in err_msg or 'invalid syntax' in err_msg:
                error_type = 'FOL_SYNTAX_ERROR'
            else:
                error_type = 'Z3_COMPILATION_ERROR'
                
        # Fallback logic: overwrite LLM direct answer if Z3 verified successfully, else fallback to LLM's own answer
        if is_z3_success and str(z3_result).strip() != "" and str(z3_result).lower() != "unknown":
            final_predicted_answer = z3_result
        else:
            final_predicted_answer = llm_ans
            
        is_correct = (str(final_predicted_answer).strip().lower() == str(gt_ans).strip().lower())
        is_direct_correct = (str(llm_ans).strip().lower() == str(gt_ans).strip().lower())
        
        if is_z3_success and not (str(z3_result).strip().lower() == str(gt_ans).strip().lower()):
            # Mark logical proof failure based on Z3 execution correctness
            error_type = 'LOGICAL_PROOF_FAILURE'
            
        premises_used_match = (sorted(llm_used) == sorted(gt_used))
        
        if gt_used:
            intersection = len(set(llm_used) & set(gt_used))
            precision = intersection / len(llm_used) if llm_used else 0.0
            recall = intersection / len(gt_used)
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        else:
            f1 = 1.0 if not llm_used else 0.0
            
        # Explanation evaluation: soft lexical similarity
        if fuzz is not None and llm_exp and gt_exp:
            exp_similarity = fuzz.token_sort_ratio(llm_exp, gt_exp) / 100.0
        else:
            exp_similarity = 1.0 if (not llm_exp and not gt_exp) else 0.0
            
        # Explanation evaluation: premise citation grounding F1
        def extract_citations(text: str) -> list:
            nums = []
            for word in re.findall(r"\b\d+\b", text):
                try:
                    val = int(word)
                    if val >= 1:
                        nums.append(val)
                except ValueError:
                    pass
            return list(set(nums))
            
        llm_citations = extract_citations(llm_exp)
        if gt_used:
            cite_intersect = len(set(llm_citations) & set(gt_used))
            cite_prec = cite_intersect / len(llm_citations) if llm_citations else 0.0
            cite_rec = cite_intersect / len(gt_used)
            citation_f1 = 2 * cite_prec * cite_rec / (cite_prec + cite_rec) if (cite_prec + cite_rec) > 0 else 0.0
        else:
            citation_f1 = 1.0 if not llm_citations else 0.0
            
        question_results.append({
            'record_idx': res['idx'],
            'question_id': q_id,
            'gt_answer': gt_ans,
            'gt_premises_used': gt_used,
            'llm_direct_answer': llm_ans,
            'llm_premises_used': llm_used,
            'llm_z3_code': llm_z3,
            'z3_result': z3_result,
            'is_z3_success': is_z3_success,
            'final_predicted_answer': final_predicted_answer,
            'is_correct': is_correct,
            'is_direct_correct': is_direct_correct,
            'error_type': error_type,
            'premises_used_match': premises_used_match,
            'premises_used_f1': f1,
            'fol_exact_match': fol_exact_match,
            'fol_similarity': fol_similarity,
            'explanation_similarity': exp_similarity,
            'explanation_citation_f1': citation_f1
        })

# Calculate Metrics
print("\n--- Calculating Final Performance Metrics ---")
total_questions = len(question_results)
total_records = len(results)

if total_questions == 0:
    print("No questions were evaluated.")
    summary = {}
else:
    xml_parse_success_count = sum(1 for q in question_results if q['error_type'] != 'XML_PARSING_ERROR')
    z3_success_count = sum(1 for q in question_results if q['is_z3_success'])
    correct_count = sum(1 for q in question_results if q['is_correct'])
    direct_correct_count = sum(1 for q in question_results if q['is_direct_correct'])
    premises_used_match_count = sum(1 for q in question_results if q['premises_used_match'])
    avg_premises_used_f1 = sum(q['premises_used_f1'] for q in question_results) / total_questions
    avg_exp_similarity = sum(q['explanation_similarity'] for q in question_results) / total_questions
    avg_exp_citation_f1 = sum(q['explanation_citation_f1'] for q in question_results) / total_questions
    
    fol_exact_match_count = sum(1 for r in results if r['llm_parsed'].get('premises_fol') == r['gt_parsed'].get('premises_fol'))
    
    sum_fol_similarity = 0.0
    for r in results:
        gt_fol = r['gt_parsed'].get('premises_fol', [])
        llm_fol = r['llm_parsed'].get('premises_fol', [])
        exact = (gt_fol == llm_fol)
        if fuzz is not None:
            sim = fuzz.token_sort_ratio("\n".join(llm_fol), "\n".join(gt_fol))
        else:
            sim = 100.0 if exact else 0.0
        sum_fol_similarity += sim
    avg_fol_similarity = sum_fol_similarity / total_records if total_records > 0 else 0.0
    
    from collections import Counter
    error_counts = Counter(q['error_type'] for q in question_results)
    
    summary = {
        'total_questions_evaluated': total_questions,
        'total_records_evaluated': total_records,
        'xml_parsing_rate': xml_parse_success_count / total_questions,
        'z3_compilation_rate': z3_success_count / total_questions,
        'pipeline_accuracy': correct_count / total_questions,
        'direct_prediction_accuracy': direct_correct_count / total_questions,
        'fol_exact_match_rate': fol_exact_match_count / total_records if total_records > 0 else 0,
        'premises_used_match_rate': premises_used_match_count / total_questions,
        'avg_premises_used_f1': avg_premises_used_f1,
        'avg_fol_similarity': avg_fol_similarity / 100.0,
        'avg_explanation_similarity': avg_exp_similarity,
        'avg_explanation_citation_f1': avg_exp_citation_f1,
        'error_distribution': dict(error_counts)
    }
    
    print('\n' + '=' * 60)
    print('UNIFIED MODEL EVALUATION SUMMARY')
    print('=' * 60)
    print(f'Total Records Evaluated          : {total_records}')
    print(f'Total Questions Evaluated        : {total_questions}')
    print(f'XML Parsing Rate                 : {summary["xml_parsing_rate"] * 100:.2f}%')
    print(f'Z3 Compilation Rate              : {summary["z3_compilation_rate"] * 100:.2f}%')
    print(f'Pipeline Accuracy (Z3 vs GT)     : {summary["pipeline_accuracy"] * 100:.2f}%')
    print(f'Direct Prediction Accuracy (Raw) : {summary["direct_prediction_accuracy"] * 100:.2f}%')
    print(f'FOL Exact Match Rate (Record)    : {summary["fol_exact_match_rate"] * 100:.2f}%')
    print(f'Premises Used Match Rate         : {summary["premises_used_match_rate"] * 100:.2f}%')
    print(f'Average Premises Used F1-Score   : {summary["avg_premises_used_f1"] * 100:.2f}%')
    print(f'Average FOL Soft Similarity      : {summary["avg_fol_similarity"] * 100:.2f}%')
    print(f'Average Explanation Similarity   : {summary["avg_explanation_similarity"] * 100:.2f}%')
    print(f'Average Explanation Citation F1  : {summary["avg_explanation_citation_f1"] * 100:.2f}%')
    print('-' * 60)
    print('ERROR & DIAGNOSTIC DISTRIBUTION')
    print('-' * 60)
    for err_name, count in error_counts.items():
        print(f'{err_name:<28} : {count} ({count/total_questions*100:.2f}%)')
    print('=' * 60)

    # Save output log
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)
    output_data = {
        'summary': summary,
        'results': results,
        'question_results': question_results
    }
    with open(args.output_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    print(f"Evaluation report successfully written to {args.output_file}!")

Loading dataset from Hugging Face Hub: NguyenAn05/exact_2026_model1_fol_translator...
Loaded 170 test samples successfully.
Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model Qwen/Qwen2.5-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading LoRA adapter from Hugging Face Hub: NguyenAn05/qwen2.5-type1-sft-lora...


...


--- Running Model Inference (Batch Size = 16) ---


Inference: 100%|██████████| 11/11 [08:57<00:00, 48.84s/it]



Freeing GPU memory...

--- Running Z3 Symbolic Verification ---


Verification: 100%|██████████| 170/170 [00:00<00:00, 633.37it/s]


--- Calculating Final Performance Metrics ---

UNIFIED MODEL EVALUATION SUMMARY
Total Records Evaluated          : 170
Total Questions Evaluated        : 170
XML Parsing Rate                 : 100.00%
Z3 Compilation Rate              : 91.76%
Pipeline Accuracy (Z3 vs GT)     : 61.18%
Direct Prediction Accuracy (Raw) : 68.82%
FOL Exact Match Rate (Record)    : 38.82%
Premises Used Match Rate         : 44.12%
Average Premises Used F1-Score   : 71.06%
Average FOL Soft Similarity      : 88.16%
Average Explanation Similarity   : 72.27%
Average Explanation Citation F1  : 65.10%
------------------------------------------------------------
ERROR & DIAGNOSTIC DISTRIBUTION
------------------------------------------------------------
SUCCESS                      : 92 (54.12%)
Z3_COMPILATION_ERROR         : 14 (8.24%)
LOGICAL_PROOF_FAILURE        : 64 (37.65%)
Evaluation report successfully written to results/logic_eval_results.json!


In [7]:
# Load and display evaluation summary report
import os
import json
report_path = "results/logic_eval_results.json"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    try:
        import pandas as pd
        summary = data.get("summary", {})
        df = pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])
        df["Value"] = df["Value"].apply(lambda x: f"{x*100:.2f}%" if isinstance(x, float) and x <= 1.0 else x)
        print(df.to_string(index=False))
    except ImportError:
        print(json.dumps(data.get("summary", {}), indent=2, ensure_ascii=False))
else:
    print(f"No evaluation report found at {report_path}. Please check if the evaluation script ran successfully.")

                     Metric                                                                    Value
  total_questions_evaluated                                                                      170
    total_records_evaluated                                                                      170
           xml_parsing_rate                                                                  100.00%
        z3_compilation_rate                                                                   91.76%
          pipeline_accuracy                                                                   61.18%
 direct_prediction_accuracy                                                                   68.82%
       fol_exact_match_rate                                                                   38.82%
   premises_used_match_rate                                                                   44.12%
       avg_premises_used_f1                                                                